# WACV 2027 Workshop on Real-World Surveillance (RWS) — Track 2: ABPR
## EXP-003: ConvNeXt-Base High-Resolution (384x192) Dual-GPU Training

> **Target Hardware:** Kaggle GPU T4 x 2 (32GB VRAM Total, 30GB CPU RAM)
> **Objective:** Overcome the -0.1252 BatchNorm domain gap, surpass MaxwellEQ (0.6280), and achieve >= 0.6500 mADM.
> **Execution Mode:** Background Run via **Save Version -> Save & Run All (Commit)** (Up to 12 Hours).


In [ ]:
# ==============================================================================
# 1. MASTER HYPERPARAMETER CONFIGURATION (Easily Editable)
# ==============================================================================
import os

CONFIG = {
    "EXPERIMENT_ID": "EXP-003",
    "MODEL_BACKBONE": "convnext_base",        # LayerNorm domain invariant backbone
    "IMAGE_SIZE": (384, 192),                 # Height, Width (+125% pixels over 256x128)
    "BATCH_SIZE": 64,                         # 32 per T4 GPU on 2x T4 setup
    "VAL_BATCH_SIZE": 128,                   # 64 per T4 GPU
    "EPOCHS": 30,                             # Full multi-epoch convergence
    "LEARNING_RATE": 1e-4,                    # Initial AdamW learning rate
    "MIN_LR": 1e-6,                           # Cosine annealing minimum
    "WEIGHT_DECAY": 1e-4,
    "ASL_GAMMA_NEG": 4.0,                    # Asymmetric Loss negative focusing
    "ASL_GAMMA_POS": 0.0,                    # Positive focusing
    "CATEGORY_GROUP_WEIGHT": 0.5,             # 5-Group anatomical softmax weight
    "LOG_INTERVAL": 25,                       # Batch logging frequency
    "DATA_DIR": "/kaggle/input/upar-challenge-2027-data/data", # Path to attached dataset
    "CHECKPOINT_DIR": "/kaggle/working/checkpoints",
    "OUTPUT_DIR": "/kaggle/working/submissions",
}

print("Config loaded successfully for", CONFIG["EXPERIMENT_ID"])

In [ ]:
# ==============================================================================
# 2. HARDWARE & DATASET VERIFICATION
# ==============================================================================
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
num_gpus = torch.cuda.device_count()
print(f"Detected GPUs: {num_gpus}")
for i in range(num_gpus):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} | Total Memory: {props.total_memory / (1024**3):.2f} GB")

# Verify dataset directory
data_dir = CONFIG["DATA_DIR"]
if not os.path.exists(data_dir):
    # Fallback check if dataset is rooted without /data
    alt_dir = "/kaggle/input/upar-challenge-2027-data"
    if os.path.exists(os.path.join(alt_dir, "annotations")):
        data_dir = alt_dir
        CONFIG["DATA_DIR"] = alt_dir
print(f"Dataset Path: {data_dir} (Exists: {os.path.exists(data_dir)})")
if os.path.exists(data_dir):
    print("Contents:", os.listdir(data_dir))


In [ ]:
# ==============================================================================
# 3. PRIVATE GITHUB REPOSITORY INGESTION (Auto-Clone / Pull)
# ==============================================================================
# You can either paste your GitHub Personal Access Token (PAT) below,
# or add it as a Kaggle Secret with label GITHUB_PAT
try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    GITHUB_TOKEN = user_secrets.get_secret("GITHUB_PAT")
except Exception:
    GITHUB_TOKEN = "YOUR_GITHUB_PAT_TOKEN_HERE"

REPO_URL = f"https://{GITHUB_TOKEN}@github.com/Aditya-Lingam-9000/upar_challenge_26_rws27.git"
REPO_DIR = "/kaggle/working/upar_challenge_26_rws27"

if os.path.exists(REPO_DIR):
    print(f"Repository directory exists. Pulling latest commits from main...")
    os.system(f"cd {REPO_DIR} && git pull origin main")
else:
    print(f"Cloning private repository from GitHub...")
    os.system(f"git clone {REPO_URL} {REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Active Directory: {os.getcwd()}")
!pip install -q -r requirements.txt


In [ ]:
# ==============================================================================
# 4. LAUNCH DUAL-GPU HIGH-RESOLUTION TRAINING (Real-Time Continuous Logs)
# ==============================================================================
!python -m src.train \
    --model {CONFIG["MODEL_BACKBONE"]} \
    --data_root {CONFIG["DATA_DIR"]} \
    --epochs {CONFIG["EPOCHS"]} \
    --batch_size {CONFIG["BATCH_SIZE"]} \
    --val_batch_size {CONFIG["VAL_BATCH_SIZE"]} \
    --height {CONFIG["IMAGE_SIZE"][0]} \
    --width {CONFIG["IMAGE_SIZE"][1]} \
    --lr {CONFIG["LEARNING_RATE"]} \
    --min_lr {CONFIG["MIN_LR"]} \
    --weight_decay {CONFIG["WEIGHT_DECAY"]} \
    --asl_weight 1.0 \
    --group_weight {CONFIG["CATEGORY_GROUP_WEIGHT"]} \
    --gamma_neg {CONFIG["ASL_GAMMA_NEG"]} \
    --gamma_pos {CONFIG["ASL_GAMMA_POS"]} \
    --exp_id {CONFIG["EXPERIMENT_ID"]} \
    --save_dir {CONFIG["CHECKPOINT_DIR"]} \
    --log_interval {CONFIG["LOG_INTERVAL"]}


In [ ]:
# ==============================================================================
# 5. VERIFY SUBMISSION ARCHIVE & EXPORT FOR CODABENCH UPLOAD
# ==============================================================================
import shutil

sub_zip = f"submissions/{CONFIG['EXPERIMENT_ID']}_submission.zip"
dest_zip = f"/kaggle/working/{CONFIG['EXPERIMENT_ID']}_submission.zip"

if os.path.exists(sub_zip):
    shutil.copy(sub_zip, dest_zip)
    size_mb = os.path.getsize(dest_zip) / (1024 * 1024)
    print(f"SUCCESS: Standalone Codabench submission package ready!")
    print(f"Location: {dest_zip} ({size_mb:.2f} MB)")
    print("You can now download this zip file directly from the Kaggle Output tab and submit to Codabench!")
else:
    print(f"Checking checkpoint: {CONFIG['CHECKPOINT_DIR']}/{CONFIG['EXPERIMENT_ID']}_best.pth")
    best_pth = f"{CONFIG['CHECKPOINT_DIR']}/{CONFIG['EXPERIMENT_ID']}_best.pth"
    if os.path.exists(best_pth):
        from src.submission_generator import package_submission
        package_submission(best_pth, dest_zip, model_name=CONFIG["MODEL_BACKBONE"], height=CONFIG["IMAGE_SIZE"][0], width=CONFIG["IMAGE_SIZE"][1])
        print(f"Packaged submission to {dest_zip} ({os.path.getsize(dest_zip)/(1024*1024):.2f} MB)")
